# So sánh CPU và GPU trên cùng snapshot/fold

Chỉ ghép hai run có cùng snapshot, fold, horizon và target. Đọc artifact đã lưu trên Drive; notebook này không huấn luyện model và không chấm final test.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
storage = Path("/content/drive/MyDrive/retail-sales-forecasting-research/screening_runs")
available = []
for path in sorted(storage.glob("*/manifest.json")):
    item = json.loads(path.read_text(encoding="utf-8"))
    available.append({"run_id": item["run_id"], "runtime": item["config"]["runtime"],
                      "folds": ",".join(item["config"]["folds"]),
                      "snapshot_id": item["source"]["snapshot_id"],
                      "models": ",".join(item["config"]["models"])})
display(pd.DataFrame(available))


## Chọn hai run

Điền run_id từ bảng trên: một CPU, một GPU. Không ghép pilot F5 với CV năm fold.

In [ ]:
CPU_RUN_ID = ""
GPU_RUN_ID = ""
if not CPU_RUN_ID or not GPU_RUN_ID:
    raise ValueError("Điền CPU_RUN_ID và GPU_RUN_ID trong cell này.")
runs = {}
for kind, run_id in [("cpu", CPU_RUN_ID), ("gpu", GPU_RUN_ID)]:
    folder = storage / run_id
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    if manifest["config"]["runtime"] != kind:
        raise ValueError(f"{run_id} không phải run {kind}.")
    runs[kind] = (folder, manifest)
a, b = runs["cpu"][1], runs["gpu"][1]
for field in ("snapshot_sha256",):
    if a[field] != b[field]:
        raise ValueError("Hai run dùng snapshot khác nhau.")
for field in ("folds", "horizons", "targets", "seed"):
    if a["config"][field] != b["config"][field]:
        raise ValueError("Hai run có " + field + " khác nhau.")
frames = []
for kind, (folder, _) in runs.items():
    item = pd.read_csv(folder / "metrics.csv")
    if kind == "gpu":
        item = item[~item["model_id"].isin(["snaive_year", "snaive_two_year"])]
    item["compute_group"] = kind
    item["artifact_root"] = str(folder)
    frames.append(item)
metrics = pd.concat(frames, ignore_index=True)
print("Tổng nhiệm vụ:", len(metrics), "| lỗi:", int((metrics.status != "success").sum()))
display(metrics[metrics.status != "success"][
    [c for c in ("compute_group","fold_id","target","horizon","model_id","error_type","error")
     if c in metrics.columns]])


## Bảng xếp hạng development

Chỉ xếp hạng model đủ mọi fold; MAE trong cùng target/horizon. MASE giúp xem thêm khi gộp nhiều target. Thời gian CPU/GPU được ghi riêng, không diễn giải là tốc độ kiến trúc trên cùng phần cứng.

In [ ]:
ok = metrics[metrics["status"] == "success"].copy()
expected = len(a["config"]["folds"])
summary = (ok.groupby(["target","horizon","model_id","compute_group"], as_index=False)
           .agg(folds=("fold_id","nunique"), mae=("mae","mean"),
                mase=("mase","mean"), mape=("mape","mean"),
                mape_coverage=("mape_coverage","mean"), rse=("rse","mean"), fit_seconds=("fit_seconds","mean"),
                predict_seconds=("predict_seconds","mean")))
summary["complete"] = summary["folds"] == expected
for (target, horizon), group in summary.groupby(["target","horizon"]):
    print(f"{target}, H={horizon}: MAE thấp hơn tốt hơn; đủ {expected} fold")
    display(group.sort_values(["complete","mae"], ascending=[False,True]))


## Biểu đồ F5 cho model hoàn thành

Mỗi biểu đồ hiển thị actual và tối đa năm model có MAE thấp nhất tại F5. Tên model và MAE có trong chú giải.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
for target in a["config"]["targets"]:
    for horizon in a["config"]["horizons"]:
        rows = ok[(ok.target == target) & (ok.horizon == horizon) &
                  (ok.fold_id == "F5")].sort_values("mae").head(5)
        if rows.empty:
            continue
        fig, ax = plt.subplots(figsize=(13, 4))
        first = True
        for row in rows.itertuples():
            path = Path(row.artifact_root) / f"F5_{target}_{horizon}_{row.model_id}" / "predictions.csv"
            pred = pd.read_csv(path, parse_dates=["ds"])
            if first:
                ax.plot(pred.ds, pred.y_true, color="black", linewidth=2, label="Actual")
                first = False
            ax.plot(pred.ds, pred.y_pred, label=f"{row.model_id} ({row.compute_group}) | MAE {row.mae:,.0f}")
        ax.set_title(f"{target} | H={horizon} | F5 development")
        ax.set_ylabel("Giá trị tiền theo nguồn")
        ax.grid(alpha=0.2)
        ax.legend()
        fig.tight_layout()
        display(fig)
        plt.close(fig)
